# Portrait Animator on Colab

Runs the whole Flask app + LivePortrait on a Colab T4 GPU and exposes it via a public Cloudflare tunnel.

**Before running:** set **Runtime → Change runtime type → T4 GPU**.

Two ways to get the project into Colab:
- **A. GitHub:** push your local `ArtMLfinal` to a GitHub repo and set `REPO_URL` below.
- **B. Upload:** zip the project locally, upload `app.zip` via the file sidebar, and set `USE_UPLOADED_ZIP=True`.

Then run every cell top-to-bottom.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/ArtMLfinal.git"  # <-- edit me
USE_UPLOADED_ZIP = False   # set True if you uploaded app.zip instead

In [ ]:
# 1. Verify you got a GPU and CUDA-enabled torch.
!nvidia-smi | head -10
import torch
print('torch', torch.__version__, 'cuda_available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA. Runtime → Change runtime type → T4 GPU, then rerun.'

In [ ]:
# 2. Pull in the project.
import os, shutil
os.chdir('/content')

if USE_UPLOADED_ZIP:
    assert os.path.exists('/content/app.zip'), 'Upload app.zip first.'
    shutil.rmtree('/content/ArtMLfinal', ignore_errors=True)
    os.makedirs('/content/ArtMLfinal', exist_ok=True)
    !cd /content/ArtMLfinal && unzip -q /content/app.zip
else:
    shutil.rmtree('/content/ArtMLfinal', ignore_errors=True)
    !git clone $REPO_URL /content/ArtMLfinal

os.chdir('/content/ArtMLfinal/app')
print('Working in:', os.getcwd())

In [ ]:
# 3. Clone LivePortrait into app/.
!if [ ! -d liveportrait ]; then git clone https://github.com/KwaiVGI/LivePortrait.git liveportrait; fi

In [ ]:
# 4. Install deps with `uv` (10-100x faster than pip and tolerant of
#    Colab's pre-installed package conflicts — those numpy>=2 warnings are
#    about jax/cupy/shap etc., which we don't use. Safe to ignore).
#
#    --system installs into Colab's system Python (same env as the notebook).
!pip install --quiet uv

# App deps first.
!uv pip install --system --no-cache-dir -r requirements.txt

# LivePortrait's Linux/CUDA deps. We deliberately skip re-installing torch
# (Colab's preinstalled CUDA torch is what we want) and numpy==1.26.4 pin
# (our app's `numpy<2` pin already satisfies LP). uv installs in parallel
# and doesn't waste time backtracking over the jax/cupy conflicts.
!uv pip install --system --no-cache-dir -r liveportrait/requirements.txt

In [ ]:
# 5. Download pretrained weights from HuggingFace (~2 GB, 1-3 min).
!uv pip install --system --quiet --upgrade 'huggingface_hub[cli]'
!mkdir -p liveportrait/pretrained_weights
!hf download KwaiVGI/LivePortrait --local-dir liveportrait/pretrained_weights

In [ ]:
# 6. Install cloudflared for the public tunnel.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 7. Quick GPU sanity check: load LivePortrait and confirm the device.
#    If this prints anything other than 'cuda', the rest will be slow.
import sys, os
sys.path.insert(0, '/content/ArtMLfinal/app')
sys.path.insert(0, '/content/ArtMLfinal/app/liveportrait')
import liveportrait_runner
pipeline = liveportrait_runner._load_pipeline()
print('LivePortrait device:', pipeline.live_portrait_wrapper.device)

In [ ]:
# 8. Start Flask in the background, then start the Cloudflare tunnel.
#    Stop both by interrupting this cell.
import subprocess, time, re, sys, os

os.environ['PORT'] = '5001'

subprocess.run(['pkill', '-f', 'app.py'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(1)

flask_log = open('/content/flask.log', 'w')
flask = subprocess.Popen(
    ['python', 'app.py'],
    stdout=flask_log, stderr=subprocess.STDOUT,
    cwd='/content/ArtMLfinal/app',
)
print('Flask PID:', flask.pid, '(logs: /content/flask.log)')

for _ in range(30):
    time.sleep(1)
    r = subprocess.run(['curl', '-s', '-o', '/dev/null', '-w', '%{http_code}',
                        'http://127.0.0.1:5001/health'], capture_output=True, text=True)
    if r.stdout.strip() == '200':
        print('Flask up.')
        break
else:
    print('Flask did not come up — check /content/flask.log')
    raise SystemExit

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5001', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
print('Waiting for tunnel URL...')
public_url = None
for line in tunnel.stdout:
    sys.stdout.write(line)
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m and not public_url:
        public_url = m.group(0)
        print('\n' + '=' * 60)
        print('OPEN THIS URL IN YOUR BROWSER:')
        print(public_url)
        print('=' * 60 + '\n')

## Notes

- The public URL (`https://…trycloudflare.com`) works from any browser.
- On a T4, a 3-second clip takes ~10-30 seconds.
- Stop by interrupting the running cell. To fully free the GPU, **Runtime → Disconnect and delete runtime**.
- Free Colab disconnects after idle. Anything under `/content/` is lost on disconnect — download any output GIFs first, or mount Google Drive.
- **About those pip dependency-conflict warnings**: they're about Colab-preinstalled packages (`jax`, `cupy`, `shap`, etc.) that want `numpy>=2`. LivePortrait pins `numpy==1.26.4`. We don't use those other packages, so the conflicts are cosmetic. `uv` installs without wasting time trying to resolve them.